In [1]:
level_depth=0

In [2]:
from IPython.display import display
import os
import numpy as np
import pandas as pd

import plotly.express as px  # per il grafico lineare


In [3]:
#STRATEGY 1

'''
Tables in DB: [('chats',)]
number of unique chats 127141
chats.db - Tabella 'chats'
           type_and_id                   token                  parent  \
0                 None  [keyword] thedemocrats                    None   
40  channel_1889806290         thedemocratskmf  [keyword] thedemocrats   
41  channel_1413288788       thedemocratsindia  [keyword] thedemocrats      

       timestamp  
0   1.722583e+09  
40  1.728093e+09  
41  1.728093e+09  

✅ discovery_edges.csv.gz, 
Il timestamp da l'ultima volta che hanno visitato quel gruppo ma questo significa che non è davvero indicativo di una timeline 

          type_and_id              parent     timestamp
0  channel_1306559115  channel_1840578235  1.722586e+09
1  channel_2036850729  channel_1840578235  1.722586e+09
2  channel_1941222046  channel_1840578235  1.722586e+09

number of non unique first nodes 284
✅ first_nodes.csv.gz
          type_and_id                    token                      parent
0  channel_2036421633               trump2024e         [keyword] Trump2024
1  channel_2178554925  biden_has_left_the_chat             [keyword] Biden
2  channel_2095394414             speech_biden      [keyword] Joseph Biden

number of unique first nodes 247
'''

#paths
path_summary_filtered = f"../results/levels/level_{level_depth}/percentage_of_politics_msgs/summary_pol_sorted_filtered_spam.csv.gz"
path_df_edges = f"../material/discovery_edges.csv.gz"

# load summary with pol_ratio_without_spam for every channel_id
summary_filtered  = pd.read_csv(path_summary_filtered, compression="gzip")
print(f"len summary_filtered = {len(summary_filtered)}")
print(f"summary_filtered:\n {summary_filtered.head()}")

# load discovery_edges
df_edges = pd.read_csv(path_df_edges, compression="gzip")
print(f"df_edges:\n {df_edges.head()}")

#filter for only the istances of channels in summary_filtered
children = df_edges[df_edges['parent'].isin(summary_filtered['channel_id'])].reset_index(drop=True)

#extract the unique istances of type_and_id
df_unique_children = children[['type_and_id']].drop_duplicates().reset_index(drop=True)

#save
print(f"lenght of df_unique_childen {len(df_unique_children)}")





len summary_filtered = 43
summary_filtered:
            channel_id  total_msgs_without_spam  pol_msgs  economy_msgs  \
0  channel_1076871110                      555       211            20   
1  channel_1245638927                      914       403            26   
2  channel_1269157403                      142        36             3   
3  channel_1283801046                       19         7             0   
4  channel_1292024994                       78        41             2   

   crypto_msgs  not_categorized_msgs  spam_msgs  outliers_msgs  \
0           18                    93      28273            213   
1           77                   180        520            228   
2            1                    74          0             28   
3            0                     6          0              6   
4            2                     6          0             27   

   total_msgs_with_spam  pol_ratio_without_spam  economy_ratio_without_spam  \
0                 28828           

In [4]:
# --- FIX: definisci i candidati e il denominatore corretto --------------------
candidate_parents = set(parents_and_children_pool['parent_num'].tolist())
candidate_edges = edges_with_child[edges_with_child['parent_num'].isin(candidate_parents)].copy()

unique_children = candidate_edges[['child_num']].drop_duplicates()
pol_msgs_per_child = (child_stats.set_index('child_num')['pol_msgs']
                      .reindex(unique_children['child_num'])
                      .fillna(0))

# Denominatore: SOLO i figli sotto i genitori candidati
global_total_pol_msgs = float(pol_msgs_per_child.sum())
if global_total_pol_msgs == 0:
    raise ValueError("Global total of political messages (candidati) è zero; coverage non definibile.")

# --- 5) Frazioni sull'asse X --------------------------------------------------
fractions = np.linspace(0.1, 1.0, 10)
total_parents = len(parents_and_children_pool)
k_counts = sorted({max(1, int(round(total_parents * f))) for f in fractions})

# --- NUOVO: sampling per fare boxplot ----------------------------------------
num_trials = 800            # ↑ se vuoi box più "stabili"
WEIGHTED = False            # True per campionare pesando con parent_pol_ratio
rng = np.random.default_rng(0)  # riproducibile

parent_list = parents_and_children_pool['parent_num'].to_numpy()
if WEIGHTED:
    weights = parents_and_children_pool['parent_pol_ratio'].to_numpy().astype(float)
    weights = weights / weights.sum() if weights.sum() > 0 else None
else:
    weights = None

box_rows = []
for k in k_counts:
    frac_pct = k / total_parents * 100.0
    for t in range(num_trials):
        chosen_parents = rng.choice(parent_list, size=k, replace=False, p=weights)
        # figli esplorati e non
        explored_mask = candidate_edges['parent_num'].isin(chosen_parents)
        explored_children = candidate_edges.loc[explored_mask, 'child_num'].unique()

        explored_share = float(pol_msgs_per_child.reindex(explored_children).sum() / global_total_pol_msgs * 100.0)
        unexplored_share = 100.0 - explored_share

        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': k,
            'trial': t,
            'Group': 'Explored children',
            'coverage_pct': explored_share
        })
        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': k,
            'trial': t,
            'Group': 'Unexplored children',
            'coverage_pct': unexplored_share
        })

coverage_box = pd.DataFrame(box_rows)

# --- Boxplot ------------------------------------------------------------------
fig = px.box(
    coverage_box,
    x='parents_explored_frac',
    y='coverage_pct',
    color='Group',
    points=False,
    labels={
        'parents_explored_frac': 'Fraction of parents explored (%)',
        'coverage_pct': 'Coverage of children political msgs (%)'
    }
)
fig.update_layout(
    title='parents choosen randomly',
    boxmode='group',
    xaxis=dict(tickformat='.0f'),
    yaxis=dict(range=[0, 100])
)
fig.show()

# (opzionale) Tabella quick-look: mediane per frazione
med = (coverage_box
       .groupby(['parents_explored_frac', 'Group'])['coverage_pct']
       .median()
       .reset_index()
       .pivot(index='parents_explored_frac', columns='Group', values='coverage_pct')
       .sort_index())
med.columns.name = None
print("\nMediane per frazione (%):\n", med.round(2).to_string())

'''
Mini-esempio numerico (una X specifica)

Supponi:

global_total_pol_msgs = 1000 (somma politica di tutti i figli candidati).

total_parents = 10.

X = 30% ⇒ k = 3.

Tre trial (su 200):

Trial 1 → scelti i genitori {G2,G5,G9}. I loro figli sommano 620 messaggi politici.

Explored: 
620
/
1000
×
100
=
62
%
620/1000×100=62%

Unexplored: 
38
%
38%
→ Due Y a X=30: 62 (Explored) e 38 (Unexplored)

Trial 2 → scelti {G1,G3,G10}. Somma figli = 540.
→ Y: 54 e 46

Trial 3 → scelti {G4,G6,G7}. Somma figli = 480.
→ Y: 48 e 52

… Ripeti fino a 200 trial ⇒ a X=30% avrai 200 numeri per Explored (es. 62, 54, 48, …) e 200 per Unexplored (38, 46, 52, …).
Il boxplot riassume proprio questi insiemi.
'''


NameError: name 'parents_and_children_pool' is not defined

In [ ]:
# --- FIX: definisci i candidati e il denominatore corretto --------------------
candidate_parents = set(parents_and_children_pool['parent_num'].tolist())
candidate_edges = edges_with_child[edges_with_child['parent_num'].isin(candidate_parents)].copy()

unique_children = candidate_edges[['child_num']].drop_duplicates()
pol_msgs_per_child = (child_stats.set_index('child_num')['pol_msgs']
                      .reindex(unique_children['child_num'])
                      .fillna(0))

# Denominatore: SOLO i figli sotto i genitori candidati
global_total_pol_msgs = float(pol_msgs_per_child.sum())
if global_total_pol_msgs == 0:
    raise ValueError("Global total of political messages (candidati) è zero; coverage non definibile.")

# --- 5) Frazioni sull'asse X --------------------------------------------------
fractions = np.linspace(0.1, 1.0, 10)
total_parents = len(parents_and_children_pool)
k_counts = sorted({max(1, int(round(total_parents * f))) for f in fractions})

# --- NUOVO: sampling per fare boxplot ----------------------------------------
num_trials = 800            # ↑ se vuoi box più "stabili"
WEIGHTED = True            # True per campionare pesando con parent_pol_ratio
rng = np.random.default_rng(0)  # riproducibile

parent_list = parents_and_children_pool['parent_num'].to_numpy()

# ============== UNICA MODIFICA: pesi safe e sempre > 0 =======================
if WEIGHTED:
    w_raw = parents_and_children_pool['parent_pol_ratio'].to_numpy().astype(float)
    w_raw = np.nan_to_num(w_raw, nan=0.0)
    if w_raw.sum() > 0:
        w_norm = w_raw / w_raw.sum()
        n = w_norm.size
        uniform = np.full(n, 1.0 / n, dtype=float)
        safe_weights = 0.95 * w_norm + 0.05 * uniform   # tutti > 0
    else:
        safe_weights = None  # fallback a non pesato se tutto zero
else:
    safe_weights = None
# ==============================================================================

box_rows = []
for k in k_counts:
    frac_pct = k / total_parents * 100.0
    for t in range(num_trials):
        chosen_parents = rng.choice(parent_list, size=k, replace=False, p=safe_weights)
        # figli esplorati e non
        explored_mask = candidate_edges['parent_num'].isin(chosen_parents)
        explored_children = candidate_edges.loc[explored_mask, 'child_num'].unique()

        explored_share = float(pol_msgs_per_child.reindex(explored_children).sum() / global_total_pol_msgs * 100.0)
        unexplored_share = 100.0 - explored_share

        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': k,
            'trial': t,
            'Group': 'Explored children',
            'coverage_pct': explored_share
        })
        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': k,
            'trial': t,
            'Group': 'Unexplored children',
            'coverage_pct': unexplored_share
        })

coverage_box = pd.DataFrame(box_rows)

# --- Boxplot ------------------------------------------------------------------
fig = px.box(
    coverage_box,
    x='parents_explored_frac',
    y='coverage_pct',
    color='Group',
    points=False,
    labels={
        'parents_explored_frac': 'Fraction of parents explored (%)',
        'coverage_pct': 'Coverage of children political msgs (%)'
    }
)
fig.update_layout(
    title='parents choosen with weighted probabily considering percentage of pol msgs',
    boxmode='group',
    xaxis=dict(tickformat='.0f'),
    yaxis=dict(range=[0, 100])
)
fig.show()

# (opzionale) Tabella quick-look: mediane per frazione
med = (coverage_box
       .groupby(['parents_explored_frac', 'Group'])['coverage_pct']
       .median()
       .reset_index()
       .pivot(index='parents_explored_frac', columns='Group', values='coverage_pct')
       .sort_index())
med.columns.name = None
print("\nMediane per frazione (%):\n", med.round(2).to_string())


In [ ]:
# ===== Imports =========================================================
import re
import numpy as np
import pandas as pd
import plotly.express as px

# ----------------------------------------------------------------------
# EXPECTED INPUT DATAFRAMES
# - summary_filtered: per-channel stats with at least:
#   ['channel_id', 'pol_msgs', 'total_msgs_without_spam', 'pol_ratio_without_spam']
# - df_edges: discovery edges with at least:
#   ['parent', 'type_and_id']  where values look like 'channel_1840578235'
# ----------------------------------------------------------------------

def strip_channel_prefix(x):
    """
    Converte:
      'channel_1840578235' -> 1840578235
      ' Channel-999 '      -> 999
      'CHAT_123'           -> 123
      456789               -> 456789
    Se non trova cifre finali, ritorna NaN.
    """
    import numpy as np, re
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return int(x)
    s = str(x).strip().replace('\ufeff', '')
    s_low = s.lower()
    for pref in ('channel_', 'channel-', 'chat_', 'chat-'):
        s_low = s_low.replace(pref, '')
    if s_low.isdigit():
        return int(s_low)
    m = re.search(r'(\d+)\s*$', s_low)
    return int(m.group(1)) if m else np.nan


# --- 1) Normalizzazione ID canale --------------------------------------
summary = summary_filtered.copy()

print(f"len summary=\n{len(summary)}")

summary['channel_id_num'] = summary['channel_id'].apply(strip_channel_prefix)
bad_summary = summary['channel_id_num'].isna().sum()
if bad_summary:
    print(f"[WARN] number of NaN channel_id_num in summary df: {bad_summary}")
    summary = summary.dropna(subset=['channel_id_num']).copy()
summary['channel_id_num'] = summary['channel_id_num'].astype(int)

edges = df_edges.copy()
edges['parent_num'] = edges['parent'].apply(strip_channel_prefix)
edges['child_num']  = edges['type_and_id'].apply(strip_channel_prefix)
edges = edges.dropna(subset=['parent_num', 'child_num']).copy()
edges['parent_num'] = edges['parent_num'].astype(int)
edges['child_num']  = edges['child_num'].astype(int)

print(f"len edges=\n{len(edges)}")

# --- 2) Attacca metriche dei figli alle edges --------------------------
child_stats = (summary[['channel_id_num', 'pol_msgs', 'total_msgs_without_spam', 'pol_ratio_without_spam']]
               .rename(columns={'channel_id_num': 'child_num'}))
edges_with_child = edges.merge(child_stats, on='child_num', how='inner')

print(f"len edges_with_child=\n{len(edges_with_child)}")

# --- 3) Lista genitori candidati ordinati per "quanto sono politici" ---
parent_stats = (summary[['channel_id_num', 'pol_ratio_without_spam']]
                .rename(columns={'channel_id_num': 'parent_num',
                                 'pol_ratio_without_spam': 'parent_pol_ratio'}))
parents_and_children_pool = (edges_with_child[['parent_num']].drop_duplicates()
                .merge(parent_stats, on='parent_num', how='left')
                .dropna(subset=['parent_pol_ratio'])
                .sort_values('parent_pol_ratio', ascending=False)
                .reset_index(drop=True))

print(f"len parents_and_children_pool=\n{len(parents_and_children_pool)}")

total_parents = len(parents_and_children_pool)
if total_parents == 0:
    raise ValueError("No candidate parents available after joins/filters.")

parent_list = parents_and_children_pool['parent_num'].to_numpy()

# --- 4) Sub-grafo: edges sotto i genitori candidati --------------------
candidate_edges = edges_with_child[edges_with_child['parent_num'].isin(parent_list)].copy()

print(f"len candidate_edges=\n{len(candidate_edges)}")

if candidate_edges.empty:
    raise ValueError("No edges found under candidate parents.")

# --- 4b) Denominatore: GLOBALE (tutto summary) ------------------------
# mappa completa child -> pol_msgs (usata per i numeratori dei figli esplorati)
pol_msgs_per_child_full = child_stats.set_index('child_num')['pol_msgs'].fillna(0)

# >>> QUI IL DENOMINATORE <<< (GLOBALE, fisso per tutti i trial)
global_total_pol_msgs = float(summary['pol_msgs'].sum())
if global_total_pol_msgs == 0:
    raise ValueError("Global total of political messages is zero; coverage cannot be computed.")

# --- 5) Frazioni sull'asse X (10%..100%) -------------------------------
fractions = np.linspace(0.1, 1.0, 10)
k_counts = sorted({max(1, int(round(total_parents * f))) for f in fractions})

# --- 6) Boxplot: sampling per distribuzioni ----------------------------
num_trials = 800            # ↑ se vuoi box più "stabili"
WEIGHTED = False            # True per campionare pesando con parent_pol_ratio
rng = np.random.default_rng(0)  # riproducibile

if WEIGHTED:
    weights = parents_and_children_pool['parent_pol_ratio'].to_numpy().astype(float)
    weights = weights / weights.sum() if weights.sum() > 0 else None
else:
    weights = None

box_rows = []
for k in k_counts:
    frac_pct = k / total_parents * 100.0
    for t in range(num_trials):
        chosen_parents = rng.choice(parent_list, size=k, replace=False, p=weights)

        # Figli "esplorati" = tutti i child con almeno un edge da un genitore scelto
        explored_mask = candidate_edges['parent_num'].isin(chosen_parents)
        explored_children = candidate_edges.loc[explored_mask, 'child_num'].unique()

        # Numeratore: somma dei messaggi politici dei figli esplorati
        explored_pol = float(pol_msgs_per_child_full.reindex(explored_children).sum())

        # Coverage su denominatore GLOBALE
        explored_share = (explored_pol / global_total_pol_msgs) * 100.0
        # Clamp anti floating noise
        explored_share = max(0.0, min(100.0, explored_share))
        unexplored_share = 100.0 - explored_share

        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': int(k),
            'trial': int(t),
            'Group': 'Explored children',
            'coverage_pct': explored_share
        })
        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': int(k),
            'trial': int(t),
            'Group': 'Unexplored children',
            'coverage_pct': unexplored_share
        })

coverage_box = pd.DataFrame(box_rows)

# --- 7) Boxplot --------------------------------------------------------
fig = px.box(
    coverage_box,
    x='parents_explored_frac',
    y='coverage_pct',
    color='Group',
    points=False,
    labels={
        'parents_explored_frac': 'Fraction of parents explored (%)',
        'coverage_pct': 'Coverage of children political msgs vs GLOBAL total (%)'
    }
)
fig.update_layout(
    title='parents choosen randomly (GLOBAL denominator)',
    boxmode='group',
    xaxis=dict(tickformat='.0f'),
    yaxis=dict(range=[0, 100]),
    legend_title_text=''
)
fig.show()

# --- 8) Tabella quick-look: mediane per frazione -----------------------
med = (coverage_box
       .groupby(['parents_explored_frac', 'Group'])['coverage_pct']
       .median()
       .reset_index()
       .pivot(index='parents_explored_frac', columns='Group', values='coverage_pct')
       .sort_index())
med.columns.name = None

print("\nMediane per frazione (%):\n", med.round(2).to_string())

# --- 9) Sanity check utili ---------------------------------------------
print("\n[CHECK] Global total pol msgs (summary):", int(global_total_pol_msgs))
candidate_children = candidate_edges['child_num'].unique()
print("[CHECK] Max copribile dentro il sottografo candidati:",
      int(pol_msgs_per_child_full.reindex(candidate_children).sum()))


In [ ]:
# ===== Imports =========================================================
import re
import numpy as np
import pandas as pd
import plotly.express as px

# ----------------------------------------------------------------------
# EXPECTED INPUT DATAFRAMES
# - summary_filtered: per-channel stats with at least:
#   ['channel_id', 'pol_msgs', 'total_msgs_without_spam', 'pol_ratio_without_spam']
# - df_edges: discovery edges with at least:
#   ['parent', 'type_and_id']  where values look like 'channel_1840578235'
# ----------------------------------------------------------------------

def strip_channel_prefix(x):
    """
    Converte:
      'channel_1840578235' -> 1840578235
      ' Channel-999 '      -> 999
      'CHAT_123'           -> 123
      456789               -> 456789
    Se non trova cifre finali, ritorna NaN.
    """
    import numpy as np, re
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return int(x)
    s = str(x).strip().replace('\ufeff', '')
    s_low = s.lower()
    for pref in ('channel_', 'channel-', 'chat_', 'chat-'):
        s_low = s_low.replace(pref, '')
    if s_low.isdigit():
        return int(s_low)
    m = re.search(r'(\d+)\s*$', s_low)
    return int(m.group(1)) if m else np.nan


# --- 1) Normalizzazione ID canale --------------------------------------
summary = summary_filtered.copy()
summary['channel_id_num'] = summary['channel_id'].apply(strip_channel_prefix)
bad_summary = summary['channel_id_num'].isna().sum()
if bad_summary:
    print(f"[WARN] number of NaN channel_id_num in summary df: {bad_summary}")
    summary = summary.dropna(subset=['channel_id_num']).copy()
summary['channel_id_num'] = summary['channel_id_num'].astype(int)

edges = df_edges.copy()
edges['parent_num'] = edges['parent'].apply(strip_channel_prefix)
edges['child_num']  = edges['type_and_id'].apply(strip_channel_prefix)
edges = edges.dropna(subset=['parent_num', 'child_num']).copy()
edges['parent_num'] = edges['parent_num'].astype(int)
edges['child_num']  = edges['child_num'].astype(int)

# --- 2) Attacca metriche dei figli alle edges --------------------------
child_stats = (summary[['channel_id_num', 'pol_msgs', 'total_msgs_without_spam', 'pol_ratio_without_spam']]
               .rename(columns={'channel_id_num': 'child_num'}))
edges_with_child = edges.merge(child_stats, on='child_num', how='inner')

# --- 3) Lista genitori candidati ordinati per "quanto sono politici" ---
parent_stats = (summary[['channel_id_num', 'pol_ratio_without_spam']]
                .rename(columns={'channel_id_num': 'parent_num',
                                 'pol_ratio_without_spam': 'parent_pol_ratio'}))
parents_and_children_pool = (edges_with_child[['parent_num']].drop_duplicates()
                .merge(parent_stats, on='parent_num', how='left')
                .dropna(subset=['parent_pol_ratio'])
                .sort_values('parent_pol_ratio', ascending=False)
                .reset_index(drop=True))

total_parents = len(parents_and_children_pool)
if total_parents == 0:
    raise ValueError("No candidate parents available after joins/filters.")

parent_list = parents_and_children_pool['parent_num'].to_numpy()

# --- 4) Sub-grafo: edges sotto i genitori candidati --------------------
candidate_edges = edges_with_child[edges_with_child['parent_num'].isin(parent_list)].copy()
if candidate_edges.empty:
    raise ValueError("No edges found under candidate parents.")

# --- 4b) Denominatore: GLOBALE (tutto summary) ------------------------
# mappa completa child -> pol_msgs (usata per i numeratori dei figli esplorati)
pol_msgs_per_child_full = child_stats.set_index('child_num')['pol_msgs'].fillna(0)

# >>> QUI IL DENOMINATORE <<< (GLOBALE, fisso per tutti i trial)
global_total_pol_msgs = float(summary['pol_msgs'].sum())
if global_total_pol_msgs == 0:
    raise ValueError("Global total of political messages is zero; coverage cannot be computed.")

# --- 5) Frazioni sull'asse X (10%..100%) -------------------------------
fractions = np.linspace(0.1, 1.0, 10)
k_counts = sorted({max(1, int(round(total_parents * f))) for f in fractions})

# --- 6) Boxplot: sampling per distribuzioni (WEIGHTED robusto) ---------
num_trials = 800            # ↑ se vuoi box più "stabili"
WEIGHTED = True             # campionamento pesato con parent_pol_ratio
rng = np.random.default_rng(0)  # riproducibile

if WEIGHTED:
    # Costruisci pesi "safe" (>0) con smoothing ε, per evitare l'errore
    # "Fewer non-zero entries in p than size" quando replace=False
    w_raw = parents_and_children_pool['parent_pol_ratio'].to_numpy(dtype=float)
    w_raw = np.nan_to_num(w_raw, nan=0.0, posinf=0.0, neginf=0.0)
    if w_raw.sum() > 0:
        w_norm = w_raw / w_raw.sum()
        n = w_norm.size
        eps = 0.05  # 5% uniforme
        uniform = np.full(n, 1.0 / n, dtype=float)
        safe_weights = (1.0 - eps) * w_norm + eps * uniform
        safe_weights = safe_weights / safe_weights.sum()  # rinormalizza per stabilità
    else:
        safe_weights = None  # fallback non pesato se tutti zero
else:
    safe_weights = None

box_rows = []
for k in k_counts:
    frac_pct = k / total_parents * 100.0
    for t in range(num_trials):
        chosen_parents = rng.choice(parent_list, size=k, replace=False, p=safe_weights)

        # Figli "esplorati" = tutti i child con almeno un edge da un genitore scelto
        explored_mask = candidate_edges['parent_num'].isin(chosen_parents)
        explored_children = candidate_edges.loc[explored_mask, 'child_num'].unique()

        # Numeratore: somma dei messaggi politici dei figli esplorati
        explored_pol = float(pol_msgs_per_child_full.reindex(explored_children).sum())

        # Coverage su denominatore GLOBALE
        explored_share = (explored_pol / global_total_pol_msgs) * 100.0
        explored_share = max(0.0, min(100.0, explored_share))  # clamp anti floating noise
        unexplored_share = 100.0 - explored_share

        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': int(k),
            'trial': int(t),
            'Group': 'Explored children',
            'coverage_pct': explored_share
        })
        box_rows.append({
            'parents_explored_frac': frac_pct,
            'k_parents': int(k),
            'trial': int(t),
            'Group': 'Unexplored children',
            'coverage_pct': unexplored_share
        })

coverage_box = pd.DataFrame(box_rows)

# --- 7) Boxplot --------------------------------------------------------
fig = px.box(
    coverage_box,
    x='parents_explored_frac',
    y='coverage_pct',
    color='Group',
    points=False,
    labels={
        'parents_explored_frac': 'Fraction of parents explored (%)',
        'coverage_pct': 'Coverage of children political msgs vs GLOBAL total (%)'
    }
)
fig.update_layout(
    title='parents chosen with weighted probability (GLOBAL denominator)',
    boxmode='group',
    xaxis=dict(tickformat='.0f'),
    yaxis=dict(range=[0, 100]),
    legend_title_text=''
)
fig.show()

# --- 8) Tabella quick-look: mediane per frazione -----------------------
med = (coverage_box
       .groupby(['parents_explored_frac', 'Group'])['coverage_pct']
       .median()
       .reset_index()
       .pivot(index='parents_explored_frac', columns='Group', values='coverage_pct')
       .sort_index())
med.columns.name = None
print("\nMediane per frazione (%):\n", med.round(2).to_string())

# --- 9) Sanity check utili ---------------------------------------------
print("\n[CHECK] Global total pol msgs (summary):", int(global_total_pol_msgs))
candidate_children = candidate_edges['child_num'].unique()
print("[CHECK] Max copribile dentro il sottografo candidati:",
      int(pol_msgs_per_child_full.reindex(candidate_children).sum()))
